In [98]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy
import yellowbrick
import mlxtend
print("Python:", sys.version)
print("All imports work. sklearn version:", sklearn.__version__)


Python: 3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]
All imports work. sklearn version: 1.9.1


In [99]:
import pandas as pd

df = pd.read_csv("data/raw/mental-heath-in-tech-2016_20161114.csv")
df.shape

(1433, 63)

In [100]:
df.head()

,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",...,"If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?","If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?",What is your age?,What is your gender?,What country do you live in?,What US state or territory do you live in?,What country do you work in?,What US state or territory do you work in?,Which of the following best describes your work position?,Do you work remotely?
0,0,26-100,1.0,NaN,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,...,Not applicable to me,Not applicable to me,39,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Sometimes
1,0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,Rarely,Sometimes,29,male,United States of America,Illinois,United States of America,Illinois,Back-end Developer|Front-end Developer,Never
2,0,6-25,1.0,NaN,No,NaN,No,No,I don't know,Neither easy nor difficult,...,Not applicable to me,Not applicable to me,38,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Always
3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Sometimes,Sometimes,43,male,United Kingdom,NaN,United Kingdom,NaN,Supervisor/Team Lead,Sometimes
4,0,6-25,0.0,1.0,Yes,Yes,No,No,No,Neither easy nor difficult,...,Sometimes,Sometimes,43,Female,United States of America,Illinois,United States of America,Illinois,Executive Leadership|Supervisor/Team Lead|Dev ...,Sometimes


In [101]:
df.dtypes
df.dtypes.value_counts()

str        56
int64       4
float64     3
Name: count, dtype: int64

In [102]:
df.select_dtypes(include="number").columns

Index(['Are you self-employed?',
       'Is your employer primarily a tech company/organization?',
       'Is your primary role within your company related to tech/IT?',
       'Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?',
       'Do you have previous employers?',
       'Have you ever sought treatment for a mental health issue from a mental health professional?',
       'What is your age?'],
      dtype='str')

In [103]:
num = df.select_dtypes(include="number")
print(num.nunique())
print(num.isna().sum())

Are you self-employed?                                                                                                    2
Is your employer primarily a tech company/organization?                                                                   2
Is your primary role within your company related to tech/IT?                                                              2
Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?     2
Do you have previous employers?                                                                                           2
Have you ever sought treatment for a mental health issue from a mental health professional?                               2
What is your age?                                                                                                        53
dtype: int64
Are you self-employed?                                                                                                 

In [104]:
selfemp = num["Are you self-employed?"]
employer_tech = num["Is your employer primarily a tech company/organization?"]
coverage = num.iloc[:, 3]

print(pd.crosstab(selfemp, employer_tech.fillna(-1)))
print(pd.crosstab(selfemp, coverage.fillna(-1)))

Is your employer primarily a tech company/organization?  -1.0   0.0   1.0
Are you self-employed?                                                   
0                                                           0   263   883
1                                                         287     0     0
Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?  -1.0  \
Are you self-employed?                                                                                                        
0                                                                                                                      1146   
1                                                                                                                         0   

Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?   0.0  \
Are you self-employed?                                              

In [105]:
role_tech = num.iloc[:, 2]

print(pd.crosstab(employer_tech.fillna(-1), role_tech.fillna(-1)))

Is your primary role within your company related to tech/IT?  -1.0   0.0   1.0
Is your employer primarily a tech company/organ...                            
-1.0                                                           287     0     0
 0.0                                                             0    15   248
 1.0                                                           883     0     0


In [106]:
for i, col in enumerate(df.columns):
    print(i, col)

0 Are you self-employed?
1 How many employees does your company or organization have?
2 Is your employer primarily a tech company/organization?
3 Is your primary role within your company related to tech/IT?
4 Does your employer provide mental health benefits as part of healthcare coverage?
5 Do you know the options for mental health care available under your employer-provided coverage?
6 Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?
7 Does your employer offer resources to learn more about mental health concerns and options for seeking help?
8 Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?
9 If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:
10 Do you think that discussing a mental health disorder with your employer would have negative consequences?

In [107]:
original_columns = list(df.columns)
print(len(original_columns))

names_0_17 = [
    "self_employed",
    "company_size",
    "employer_is_tech",
    "role_is_tech",
    "cur_benefits_provided",
    "cur_knows_care_options",
    "cur_employer_discussed_mh",
    "cur_resources_offered",
    "cur_anonymity_protected",
    "cur_leave_ease",
    "cur_mh_disclosure_conseq",
    "cur_ph_disclosure_conseq",
    "cur_comfort_coworkers",
    "cur_comfort_supervisor",
    "cur_mh_equals_ph",
    "cur_observed_neg_conseq",
    "mh_coverage_self_emp",
    "knows_help_resources_self_emp",
]

print(len(names_0_17))
for i in range(18):
    print(i, names_0_17[i], "<-", original_columns[i][:60])

names_18_35 = [
    "reveal_to_clients_self_emp",
    "reveal_clients_neg_impact_self_emp",
    "reveal_to_coworkers_self_emp",
    "reveal_coworkers_neg_impact_self_emp",
    "productivity_affected_self_emp",
    "productivity_pct_affected_self_emp",
    "has_prev_employers",
    "prev_benefits_provided",
    "prev_knew_care_options",
    "prev_employer_discussed_mh",
    "prev_resources_offered",
    "prev_anonymity_protected",
    "prev_mh_disclosure_conseq",
    "prev_ph_disclosure_conseq",
    "prev_willing_coworkers",
    "prev_willing_supervisor",
    "prev_mh_equals_ph",
    "prev_observed_neg_conseq",
]

print(len(names_18_35))
for i in range(18):
    print(i + 18, names_18_35[i], "<-", original_columns[i + 18][:60])

names_36_62 = [
    "would_raise_ph_interview",
    "why_ph_interview_text",
    "would_raise_mh_interview",
    "why_mh_interview_text",
    "career_hurt_if_identified",
    "coworkers_view_negatively",
    "willing_share_friends_family",
    "observed_unsupportive_response",
    "observation_less_likely_reveal",
    "family_history_mh",
    "mh_disorder_past",
    "mh_disorder_current",
    "current_conditions_text",
    "suspected_conditions_text",
    "diagnosed_by_professional",
    "diagnosed_conditions_text",
    "sought_treatment",
    "interferes_when_treated",
    "interferes_when_not_treated",
    "age",
    "gender",
    "country_live",
    "us_state_live",
    "country_work",
    "us_state_work",
    "work_position",
    "works_remotely",
]

print(len(names_36_62))
for i in range(27):
    print(i + 36, names_36_62[i], "<-", original_columns[i + 36][:60])

new_names = names_0_17 + names_18_35 + names_36_62

df.columns = new_names

print(df.shape)
print(list(df.columns[:5]))


63
18
0 self_employed <- Are you self-employed?
1 company_size <- How many employees does your company or organization have?
2 employer_is_tech <- Is your employer primarily a tech company/organization?
3 role_is_tech <- Is your primary role within your company related to tech/IT?
4 cur_benefits_provided <- Does your employer provide mental health benefits as part of
5 cur_knows_care_options <- Do you know the options for mental health care available und
6 cur_employer_discussed_mh <- Has your employer ever formally discussed mental health (for
7 cur_resources_offered <- Does your employer offer resources to learn more about menta
8 cur_anonymity_protected <- Is your anonymity protected if you choose to take advantage 
9 cur_leave_ease <- If a mental health issue prompted you to request a medical l
10 cur_mh_disclosure_conseq <- Do you think that discussing a mental health disorder with y
11 cur_ph_disclosure_conseq <- Do you think that discussing a physical health issue with yo
12 cur

In [108]:
column_map = pd.DataFrame({
    "position": range(len(new_names)),
    "short_name": new_names,
    "original_question": original_columns,
})

print(column_map.shape)
column_map.head()

column_map.to_csv("data/column_map.csv", index=False)

check = pd.read_csv("data/column_map.csv")

(63, 3)


In [109]:
for col in ["company_size", "cur_benefits_provided", "cur_leave_ease"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

company_size
company_size
26-100            292
NaN               287
More than 1000    256
100-500           248
6-25              210
500-1000           80
1-5                60
Name: count, dtype: int64

cur_benefits_provided
cur_benefits_provided
Yes                                531
I don't know                       319
NaN                                287
No                                 213
Not eligible for coverage / N/A     83
Name: count, dtype: int64

cur_leave_ease
cur_leave_ease
NaN                           287
Somewhat easy                 281
Very easy                     220
Somewhat difficult            199
Neither easy nor difficult    178
I don't know                  150
Very difficult                118
Name: count, dtype: int64



In [110]:
scale_levels = {
    "self_employed": "binary",
    "company_size": "ordinal",
    "employer_is_tech": "binary",
    "role_is_tech": "binary",
    "cur_benefits_provided": "nominal",
    "cur_leave_ease": "ordinal",
}

print(len(scale_levels))
print(scale_levels["cur_leave_ease"])

6
ordinal


In [111]:
for col in ["cur_knows_care_options", "cur_employer_discussed_mh",
            "cur_resources_offered", "cur_anonymity_protected"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

cur_knows_care_options
cur_knows_care_options
NaN              420
No               354
I am not sure    352
Yes              307
Name: count, dtype: int64

cur_employer_discussed_mh
cur_employer_discussed_mh
No              813
NaN             287
Yes             230
I don't know    103
Name: count, dtype: int64

cur_resources_offered
cur_resources_offered
No              531
I don't know    320
Yes             295
NaN             287
Name: count, dtype: int64

cur_anonymity_protected
cur_anonymity_protected
I don't know    742
Yes             320
NaN             287
No               84
Name: count, dtype: int64



In [112]:
scale_levels.update({
    "cur_knows_care_options": "nominal",
    "cur_employer_discussed_mh": "nominal",
    "cur_resources_offered": "nominal",
    "cur_anonymity_protected": "nominal",
})

print(len(scale_levels))

10


In [113]:
for col in ["cur_mh_disclosure_conseq", "cur_ph_disclosure_conseq",
            "cur_comfort_coworkers", "cur_comfort_supervisor"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

cur_mh_disclosure_conseq
cur_mh_disclosure_conseq
Maybe    487
No       438
NaN      287
Yes      221
Name: count, dtype: int64

cur_ph_disclosure_conseq
cur_ph_disclosure_conseq
No       837
NaN      287
Maybe    268
Yes       41
Name: count, dtype: int64

cur_comfort_coworkers
cur_comfort_coworkers
Maybe    479
No       392
NaN      287
Yes      275
Name: count, dtype: int64

cur_comfort_supervisor
cur_comfort_supervisor
Yes      428
Maybe    382
No       336
NaN      287
Name: count, dtype: int64



In [114]:
scale_levels.update({
    "cur_mh_disclosure_conseq": "ordinal",
    "cur_ph_disclosure_conseq": "ordinal",
    "cur_comfort_coworkers": "ordinal",
    "cur_comfort_supervisor": "ordinal",
})

print(len(scale_levels))

14


In [115]:
for col in ["cur_mh_equals_ph", "cur_observed_neg_conseq", "knows_help_resources_self_emp"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()


cur_mh_equals_ph
cur_mh_equals_ph
I don't know    493
Yes             350
No              303
NaN             287
Name: count, dtype: int64

cur_observed_neg_conseq
cur_observed_neg_conseq
No     1048
NaN     287
Yes      98
Name: count, dtype: int64

knows_help_resources_self_emp
knows_help_resources_self_emp
NaN                     1146
I know some              141
Yes, I know several       83
No, I don't know any      63
Name: count, dtype: int64



In [116]:
scale_levels.update({
    "cur_mh_equals_ph": "nominal",
    "cur_observed_neg_conseq": "binary",
    "mh_coverage_self_emp": "binary",
    "knows_help_resources_self_emp": "ordinal",
})

print(len(scale_levels))


18


In [117]:
print(pd.crosstab(df["self_employed"], df["knows_help_resources_self_emp"].fillna("missing")))

knows_help_resources_self_emp  I know some  No, I don't know any  \
self_employed                                                      
0                                        0                     0   
1                                      141                    63   

knows_help_resources_self_emp  Yes, I know several  missing  
self_employed                                                
0                                                0     1146  
1                                               83        0  


In [118]:
for col in ["reveal_to_clients_self_emp", "reveal_clients_neg_impact_self_emp",
            "reveal_to_coworkers_self_emp", "reveal_coworkers_neg_impact_self_emp"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

reveal_to_clients_self_emp
reveal_to_clients_self_emp
NaN                                          1146
Not applicable to me                          101
No, because it would impact me negatively      83
Sometimes, if it comes up                      57
No, because it doesn't matter                  44
Yes, always                                     2
Name: count, dtype: int64

reveal_clients_neg_impact_self_emp
reveal_clients_neg_impact_self_emp
NaN             1289
I'm not sure      66
No                42
Yes               36
Name: count, dtype: int64

reveal_to_coworkers_self_emp
reveal_to_coworkers_self_emp
NaN                                          1146
Not applicable to me                          111
Sometimes, if it comes up                      99
No, because it would impact me negatively      51
No, because it doesn't matter                  15
Yes, always                                    11
Name: count, dtype: int64

reveal_coworkers_neg_impact_self_emp
reveal_coworkers

In [119]:
for col in ["reveal_to_clients_self_emp", "reveal_clients_neg_impact_self_emp",
            "reveal_to_coworkers_self_emp", "reveal_coworkers_neg_impact_self_emp"]:
    print(col)
    print(pd.crosstab(df["self_employed"], df[col].isna()))
    print()

reveal_to_clients_self_emp
reveal_to_clients_self_emp  False  True 
self_employed                           
0                               0   1146
1                             287      0

reveal_clients_neg_impact_self_emp
reveal_clients_neg_impact_self_emp  False  True 
self_employed                                   
0                                       0   1146
1                                     144    143

reveal_to_coworkers_self_emp
reveal_to_coworkers_self_emp  False  True 
self_employed                             
0                                 0   1146
1                               287      0

reveal_coworkers_neg_impact_self_emp
reveal_coworkers_neg_impact_self_emp  False  True 
self_employed                                     
0                                         0   1146
1                                       287      0



In [120]:
a = df["reveal_to_clients_self_emp"].fillna("missing").str[:20]
b = df["reveal_clients_neg_impact_self_emp"].fillna("missing").str[:20]
print(pd.crosstab(a, b))

reveal_clients_neg_impact_self_emp  I'm not sure  No  Yes  missing
reveal_to_clients_self_emp                                        
No, because it doesn                          13   7    3       21
No, because it would                          23   5   24       31
Not applicable to me                           6   4    1       90
Sometimes, if it com                          24  24    8        1
Yes, always                                    0   2    0        0
missing                                        0   0    0     1146


In [121]:
scale_levels.update({
    "reveal_to_clients_self_emp": "nominal",
    "reveal_clients_neg_impact_self_emp": "nominal",
    "reveal_to_coworkers_self_emp": "nominal",
    "reveal_coworkers_neg_impact_self_emp": "nominal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))
print(list(df.columns[18:22]))
print(list(column_map.loc[18:21, "short_name"]))

22
set()
['reveal_to_clients_self_emp', 'reveal_clients_neg_impact_self_emp', 'reveal_to_coworkers_self_emp', 'reveal_coworkers_neg_impact_self_emp']
['reveal_to_clients_self_emp', 'reveal_clients_neg_impact_self_emp', 'reveal_to_coworkers_self_emp', 'reveal_coworkers_neg_impact_self_emp']


In [122]:
for col in ["productivity_affected_self_emp", "productivity_pct_affected_self_emp"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()
    

productivity_affected_self_emp
productivity_affected_self_emp
NaN                     1146
Yes                      204
Unsure                    38
Not applicable to me      31
No                        14
Name: count, dtype: int64

productivity_pct_affected_self_emp
productivity_pct_affected_self_emp
NaN        1229
1-25%        92
26-50%       72
51-75%       26
76-100%      14
Name: count, dtype: int64



In [123]:
print(pd.crosstab(df["self_employed"], df["productivity_affected_self_emp"].isna()))
print()
pct_asked = df["productivity_pct_affected_self_emp"].notna()
print(pd.crosstab(df["productivity_affected_self_emp"].fillna("missing"), pct_asked))

productivity_affected_self_emp  False  True 
self_employed                               
0                                   0   1146
1                                 287      0

productivity_pct_affected_self_emp  False  True 
productivity_affected_self_emp                  
No                                     14      0
Not applicable to me                   31      0
Unsure                                 38      0
Yes                                     0    204
missing                              1146      0


In [124]:
scale_levels.update({
    "productivity_affected_self_emp": "nominal",
    "productivity_pct_affected_self_emp": "ordinal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

24
set()


In [125]:
print(df["has_prev_employers"].value_counts())

prev_cols = [c for c in df.columns if c.startswith("prev_")]
print(len(prev_cols))

print(df[prev_cols].isna().sum())


has_prev_employers
1    1264
0     169
Name: count, dtype: int64
11
prev_benefits_provided        169
prev_knew_care_options        169
prev_employer_discussed_mh    169
prev_resources_offered        169
prev_anonymity_protected      169
prev_mh_disclosure_conseq     169
prev_ph_disclosure_conseq     169
prev_willing_coworkers        169
prev_willing_supervisor       169
prev_mh_equals_ph             169
prev_observed_neg_conseq      169
dtype: int64


In [126]:
answered = df[prev_cols].notna().sum(axis=1)
print(pd.crosstab(df["has_prev_employers"], answered))


col_0                0     11
has_prev_employers           
0                   169     0
1                     0  1264


In [127]:
for col in ["prev_benefits_provided", "prev_knew_care_options",
            "prev_employer_discussed_mh", "prev_resources_offered"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

prev_benefits_provided
prev_benefits_provided
Some did             391
No, none did         372
I don't know         313
Yes, they all did    188
NaN                  169
Name: count, dtype: int64

prev_knew_care_options
prev_knew_care_options
N/A (not currently aware)          582
I was aware of some                384
Yes, I was aware of all of them    181
NaN                                169
No, I only became aware later      117
Name: count, dtype: int64

prev_employer_discussed_mh
prev_employer_discussed_mh
None did             890
Some did             255
NaN                  169
I don't know          86
Yes, they all did     33
Name: count, dtype: int64

prev_resources_offered
prev_resources_offered
None did             842
Some did             371
NaN                  169
Yes, they all did     51
Name: count, dtype: int64



In [128]:
scale_levels.update({
    "has_prev_employers": "binary",
    "prev_benefits_provided": "ordinal",
    "prev_knew_care_options": "nominal",
    "prev_employer_discussed_mh": "ordinal",
    "prev_resources_offered": "ordinal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

29
set()


In [129]:
for col in ["prev_anonymity_protected", "prev_mh_disclosure_conseq",
            "prev_ph_disclosure_conseq", "prev_willing_coworkers"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()
    

prev_anonymity_protected
prev_anonymity_protected
I don't know    860
NaN             169
Yes, always     164
No              121
Sometimes       119
Name: count, dtype: int64

prev_mh_disclosure_conseq
prev_mh_disclosure_conseq
Some of them        615
I don't know        310
Yes, all of them    226
NaN                 169
None of them        113
Name: count, dtype: int64

prev_ph_disclosure_conseq
prev_ph_disclosure_conseq
Some of them        631
None of them        559
NaN                 169
Yes, all of them     74
Name: count, dtype: int64

prev_willing_coworkers
prev_willing_coworkers
Some of my previous employers           740
No, at none of my previous employers    430
NaN                                     169
Yes, at all of my previous employers     94
Name: count, dtype: int64



In [130]:
scale_levels.update({
    "prev_anonymity_protected": "nominal",
    "prev_mh_disclosure_conseq": "ordinal",
    "prev_ph_disclosure_conseq": "ordinal",
    "prev_willing_coworkers": "ordinal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

33
set()


In [131]:
for col in ["prev_willing_supervisor", "prev_mh_equals_ph", "prev_observed_neg_conseq"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

prev_willing_supervisor
prev_willing_supervisor
Some of my previous employers           654
No, at none of my previous employers    416
NaN                                     169
I don't know                            101
Yes, at all of my previous employers     93
Name: count, dtype: int64

prev_mh_equals_ph
prev_mh_equals_ph
None did             463
Some did             427
I don't know         331
NaN                  169
Yes, they all did     43
Name: count, dtype: int64

prev_observed_neg_conseq
prev_observed_neg_conseq
None of them        758
Some of them        444
NaN                 169
Yes, all of them     62
Name: count, dtype: int64



In [133]:
scale_levels.update({
    "prev_willing_supervisor": "ordinal",
    "prev_mh_equals_ph": "ordinal",
    "prev_observed_neg_conseq": "ordinal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

36
set()


In [134]:
print(len(scale_levels))
print(set(scale_levels) - set(df.columns))
print(list(df.columns[22:24]))
print(list(column_map.loc[22:23, "short_name"]))

36
set()
['productivity_affected_self_emp', 'productivity_pct_affected_self_emp']
['productivity_affected_self_emp', 'productivity_pct_affected_self_emp']


In [135]:
for col in ["would_raise_ph_interview", "would_raise_mh_interview",
            "career_hurt_if_identified", "coworkers_view_negatively"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

would_raise_ph_interview
would_raise_ph_interview
Maybe    633
No       441
Yes      359
Name: count, dtype: int64

would_raise_mh_interview
would_raise_mh_interview
No       883
Maybe    438
Yes      112
Name: count, dtype: int64

career_hurt_if_identified
career_hurt_if_identified
Maybe                         588
Yes, I think it would         563
No, I don't think it would    147
Yes, it has                   105
No, it has not                 30
Name: count, dtype: int64

coworkers_view_negatively
coworkers_view_negatively
Maybe                           591
Yes, I think they would         403
No, I don't think they would    348
No, they do not                  49
Yes, they do                     42
Name: count, dtype: int64



In [136]:
scale_levels.update({
    "would_raise_ph_interview": "ordinal",
    "would_raise_mh_interview": "ordinal",
    "career_hurt_if_identified": "ordinal",
    "coworkers_view_negatively": "ordinal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

40
set()


In [137]:
for col in ["willing_share_friends_family", "observed_unsupportive_response",
            "observation_less_likely_reveal", "family_history_mh"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

willing_share_friends_family
willing_share_friends_family
Somewhat open                                            640
Very open                                                251
Somewhat not open                                        214
Neutral                                                  141
Not applicable to me (I do not have a mental illness)    112
Not open at all                                           75
Name: count, dtype: int64

observed_unsupportive_response
observed_unsupportive_response
No                    567
Maybe/Not sure        346
Yes, I observed       264
Yes, I experienced    167
NaN                    89
Name: count, dtype: int64

observation_less_likely_reveal
observation_less_likely_reveal
NaN      776
Yes      246
No       234
Maybe    177
Name: count, dtype: int64

family_history_mh
family_history_mh
Yes             670
No              488
I don't know    275
Name: count, dtype: int64



In [138]:
scale_levels.update({
    "willing_share_friends_family": "ordinal",
    "observed_unsupportive_response": "nominal",
    "observation_less_likely_reveal": "ordinal",
    "family_history_mh": "nominal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

44
set()


In [139]:
miss = df["observed_unsupportive_response"].isna()
print(pd.crosstab([df["self_employed"], df["has_prev_employers"]], miss))
print()
print(pd.crosstab(df["observed_unsupportive_response"].fillna("missing"),
                  df["observation_less_likely_reveal"].fillna("missing")))

observed_unsupportive_response    False  True 
self_employed has_prev_employers              
0             0                     118     13
              1                     976     39
1             0                      24     14
              1                     226     23

observation_less_likely_reveal  Maybe   No  Yes  missing
observed_unsupportive_response                          
Maybe/Not sure                     82  137   53       74
No                                  0    0    0      567
Yes, I experienced                 33   41   65       28
Yes, I observed                    62   56  128       18
missing                             0    0    0       89


In [140]:
for col in ["mh_disorder_past", "mh_disorder_current", "diagnosed_by_professional"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

mh_disorder_past
mh_disorder_past
Yes      736
No       451
Maybe    246
Name: count, dtype: int64

mh_disorder_current
mh_disorder_current
Yes      575
No       531
Maybe    327
Name: count, dtype: int64

diagnosed_by_professional
diagnosed_by_professional
No     717
Yes    716
Name: count, dtype: int64



In [141]:
scale_levels.update({
    "mh_disorder_past": "ordinal",
    "mh_disorder_current": "ordinal",
    "diagnosed_by_professional": "binary",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

47
set()


In [142]:
for col in ["interferes_when_treated", "interferes_when_not_treated"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

print(pd.crosstab(df["interferes_when_treated"].isna(),
                  df["interferes_when_not_treated"].isna()))

interferes_when_treated
interferes_when_treated
Not applicable to me    557
Sometimes               369
Rarely                  322
Never                   120
Often                    65
Name: count, dtype: int64

interferes_when_not_treated
interferes_when_not_treated
Often                   538
Not applicable to me    468
Sometimes               363
Rarely                   52
Never                    12
Name: count, dtype: int64

interferes_when_not_treated  False
interferes_when_treated           
False                         1433


In [143]:
scale_levels.update({
    "sought_treatment": "binary",
    "interferes_when_treated": "ordinal",
    "interferes_when_not_treated": "ordinal",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))

50
set()


In [144]:
for col in ["country_live", "us_state_live", "country_work",
            "us_state_work", "work_position", "works_remotely"]:
    print(col, "| distinct:", df[col].nunique(), "| missing:", df[col].isna().sum())
    print(df[col].value_counts().head(5))
    print()

country_live | distinct: 53 | missing: 0
country_live
United States of America    840
United Kingdom              180
Canada                       78
Germany                      58
Netherlands                  48
Name: count, dtype: int64

us_state_live | distinct: 47 | missing: 593
us_state_live
California    130
Illinois       56
Michigan       48
New York       45
Washington     43
Name: count, dtype: int64

country_work | distinct: 53 | missing: 0
country_work
United States of America    851
United Kingdom              183
Canada                       74
Germany                      58
Netherlands                  47
Name: count, dtype: int64

us_state_work | distinct: 48 | missing: 582
us_state_work
California    141
Illinois       58
New York       49
Michigan       47
Texas          44
Name: count, dtype: int64

work_position | distinct: 264 | missing: 0
work_position
Back-end Developer                        263
Front-end Developer                       125
Other              

In [145]:
us_live = df["country_live"] == "United States of America"
print(pd.crosstab(us_live, df["us_state_live"].isna()))

us_state_live  False  True 
country_live               
False              0    593
True             840      0


In [146]:
 print(df["works_remotely"].value_counts(dropna=False))

works_remotely
Sometimes    757
Always       343
Never        333
Name: count, dtype: int64


In [147]:
for col in ["us_state_work", "work_position"]:
    print(col, "| distinct:", df[col].nunique(), "| missing:", df[col].isna().sum())
    print(df[col].value_counts().head(5))
    print()

us_state_work | distinct: 48 | missing: 582
us_state_work
California    141
Illinois       58
New York       49
Michigan       47
Texas          44
Name: count, dtype: int64

work_position | distinct: 264 | missing: 0
work_position
Back-end Developer                        263
Front-end Developer                       125
Other                                     112
Supervisor/Team Lead                       68
Back-end Developer|Front-end Developer     61
Name: count, dtype: int64



In [148]:
us_work = df["country_work"] == "United States of America"
print(pd.crosstab(us_work, df["us_state_work"].isna()))

us_state_work  False  True 
country_work               
False              0    582
True             851      0


In [149]:
print("work_position | distinct:", df["work_position"].nunique(),
      "| missing:", df["work_position"].isna().sum())
print(df["work_position"].value_counts().head(5))

work_position | distinct: 264 | missing: 0
work_position
Back-end Developer                        263
Front-end Developer                       125
Other                                     112
Supervisor/Team Lead                       68
Back-end Developer|Front-end Developer     61
Name: count, dtype: int64


In [150]:
scale_levels.update({
    "country_live": "nominal",
    "us_state_live": "nominal",
    "country_work": "nominal",
    "us_state_work": "nominal",
    "works_remotely": "ordinal",
    "age": "ratio",
    "why_ph_interview_text": "free text",
    "why_mh_interview_text": "free text",
    "current_conditions_text": "free text",
    "suspected_conditions_text": "free text",
    "diagnosed_conditions_text": "free text",
    "gender": "free text",
})

print(len(scale_levels))
print(set(scale_levels) - set(df.columns))
print(set(df.columns) - set(scale_levels))

62
set()
{'work_position'}


In [151]:
scale_levels["work_position"] = "nominal"

print(len(scale_levels))
print(set(df.columns) - set(scale_levels))

roles = df["work_position"].str.split("|").explode()
print("distinct single roles:", roles.nunique())
print(roles.value_counts().head(8))
print(df["work_position"].str.count(r"\|").add(1).value_counts().sort_index())

63
set()
distinct single roles: 12
work_position
Back-end Developer      737
Front-end Developer     502
DevOps/SysAdmin         282
Supervisor/Team Lead    277
Other                   187
Support                 168
One-person shop         161
Designer                135
Name: count, dtype: int64
work_position
1     817
2     281
3     159
4     106
5      33
6      22
7       9
8       3
9       2
10      1
Name: count, dtype: int64
